In [ ]:
# run this notebook after 19_distribution_genome_figure_files_R
# use this notebook to prepare files for plotting the distribution of sibling dnms across the genome 
# make sure to download testis methylation from https://www.encodeproject.org/files/ENCFF053FUK/@@download/ENCFF053FUK.bed.gz
# make sure to download lcl replication timing from caballero and koren 2023 cell genomics data s2 (LCL.RT.csv)
# after this, run 21_distribution_genome_figure_R

In [ ]:
# annotate all cpg>tpg mutations with testis methylation level 

import gzip
import pandas as pd

# ── config ────────────────────────────────────────────────────────────────────
WGBS_FILE  = "./ENCFF053FUK.bed.gz"
CPG_FILE   = "./cpg_mutations.tsv"       # your cpg dataframe
OUTFILE    = "./cpg_with_meth.tsv"
METH_COL   = 10
AUTOSOMES  = {f"chr{i}" for i in range(1, 23)}
# ─────────────────────────────────────────────────────────────────────────────

# ── load mutations ────────────────────────────────────────────────────────────
cpg = pd.read_csv(CPG_FILE, sep="\t")
cpg = cpg[cpg["chr"].isin(AUTOSOMES)]

# ── stream wgbs file and extract methylation at mutation positions ─────────────
# index mutations by (chr, pos) for fast lookup
mut_index = set(zip(cpg["chr"], cpg["pos"] - 1))
dataset_by_pos = dict(zip(zip(cpg["chr"], cpg["pos"] - 1), cpg["dataset"]))

meth_lookup = {}  # (chr, pos) -> methylation %

open_fn = gzip.open if WGBS_FILE.endswith(".gz") else open
with open_fn(WGBS_FILE, "rt") as fh:
    for line in fh:
        if line.startswith(("#", "track", "browser")):
            continue
        cols = line.rstrip().split("\t")
        if len(cols) <= METH_COL:
            continue
        try:
            chrom = cols[0]
            pos   = int(cols[1])
            meth  = float(cols[METH_COL])
        except ValueError:
            continue
        if (chrom, pos) in mut_index:
            meth_lookup[(chrom, pos)] = meth

print(f"Matched {len(meth_lookup):,} of {len(cpg):,} mutations to WGBS data")

# ── join back to mutations ────────────────────────────────────────────────────
cpg["methylation"] = cpg.apply(lambda r: meth_lookup.get((r["chr"], r["pos"] - 1), None), axis=1)
cpg.to_csv(OUTFILE, sep="\t", index=False)
print(f"Done! Written to {OUTFILE}")

In [ ]:
# calculate denominator per cpg>tpg window 

import gzip
import bisect
import pandas as pd
from collections import defaultdict

CPG_BED  = "./ENCFF053FUK.bed.gz"
GIAB_BED = "./giab_difficult_merged_oct27.bed"
AUTOSOMES = set(["chr" + str(i) for i in range(1, 23)])  # adjust if no "chr" prefix

# ── load giab intervals ───────────────────────────────────────────────────────
giab = defaultdict(list)
with open(GIAB_BED) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("chr"):
            continue
        fields = line.split("\t")
        if not fields[1].lstrip("-").isdigit():
            continue
        chrom, start, end = fields[0], int(fields[1]), int(fields[2])
        giab[chrom].append((start, end))
for chrom in giab:
    giab[chrom].sort()

def in_giab(chrom, pos, giab):
    intervals = giab.get(chrom, [])
    if not intervals:
        return False
    starts = [s for s, e in intervals]
    idx = bisect.bisect_right(starts, pos) - 1
    if idx >= 0 and intervals[idx][0] <= pos < intervals[idx][1]:
        return True
    return False

# ── read cpg bed and filter ───────────────────────────────────────────────────
records = []
with gzip.open(CPG_BED, "rt") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        fields = line.split("\t")
        if not fields[1].lstrip("-").isdigit():
            continue
        chrom = fields[0]

        if chrom not in AUTOSOMES:   # skip chrx, chry, chrm etc.
            continue

        start = int(fields[1])
        meth  = float(fields[10])

        if in_giab(chrom, start, giab):
            continue

        records.append(meth)

# ── bin ───────────────────────────────────────────────────────────────────────
df = pd.DataFrame({"meth_pct": records})

df["bin"] = pd.cut(
    df["meth_pct"],
    bins=[0, 20, 40, 60, 80, 100],
    labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"],
    include_lowest=True
)

result = df["bin"].value_counts().sort_index().reset_index()
result.columns = ["meth_bin", "n_cpg"]
result["pct_of_total"] = result["n_cpg"] / result["n_cpg"].sum() * 100

print(result.to_string(index=False))


In [ ]:
# generate 1 mb replication timing windows  


import pandas as pd
import bisect
from collections import defaultdict
import math

# ── config ────────────────────────────────────────────────────────────────────
RT_FILE     = "./LCL.RT.csv"
OUTFILE     = "./rt_windows.bed"
WINDOW_SIZE = 1_000_000
N_BINS      = 10
AUTOSOMES   = {str(i) for i in range(1, 23)}
MIN_WINDOWS = 1  # minimum 2.5kb rt windows required to include a 1mb window
# ─────────────────────────────────────────────────────────────────────────────

# ── load rt file ──────────────────────────────────────────────────────────────
print("Loading RT file...")
rt = pd.read_csv(RT_FILE, header=None,
                 names=["chr", "start", "end", "center", "mean_rt", "male_rt", "female_rt"])
rt["chr"] = rt["chr"].astype(str)
rt = rt[rt["chr"].isin(AUTOSOMES)]

import numpy as np

# for each rt window, compute overlap with each 1mb bin
def assign_with_overlap(rt, window_size):
    # compute first and last 1mb bin each window touches
    rt["first_bin"] = (rt["start"] // window_size) * window_size
    rt["last_bin"]  = ((rt["end"] - 1) // window_size) * window_size
    
    # most rows have first_bin == last_bin (non-crossing)
    # split rows where first_bin != last_bin
    non_crossing = rt[rt["first_bin"] == rt["last_bin"]].copy()
    non_crossing["win_start"] = non_crossing["first_bin"]
    non_crossing["overlap_bp"] = non_crossing["end"] - non_crossing["start"]
    
    crossing = rt[rt["first_bin"] != rt["last_bin"]].copy()
    # for crossing windows, create one row per 1mb bin they overlap
    expanded = []
    for _, row in crossing.iterrows():
        start_bin = int(row["first_bin"])
        end_bin = int(row["last_bin"])
        for bin_start in range(start_bin, end_bin + window_size, window_size):
            overlap_start = max(row["start"], bin_start)
            overlap_end = min(row["end"], bin_start + window_size)
            if overlap_end > overlap_start:
                new_row = row.copy()
                new_row["win_start"] = bin_start
                new_row["overlap_bp"] = overlap_end - overlap_start
                expanded.append(new_row)
    
    crossing_split = pd.DataFrame(expanded) if expanded else pd.DataFrame(columns=non_crossing.columns)
    
    return pd.concat([non_crossing, crossing_split], ignore_index=True)

rt_assigned = assign_with_overlap(rt, WINDOW_SIZE)

# now aggregate with overlap-weighted mean
agg = (
    rt_assigned.groupby(["chr", "win_start"])
    .apply(lambda g: pd.Series({
        "mean_rt": np.average(g["mean_rt"], weights=g["overlap_bp"]),
        "n_rt_windows": len(g),
        "total_bp": g["overlap_bp"].sum()
    }))
    .reset_index()
)

agg = agg[agg["n_rt_windows"] >= MIN_WINDOWS]
print(f"Total 1Mb windows: {len(agg):,}")

# ── rank into n_bins by mean rt ───────────────────────────────────────────────
agg = agg.sort_values("mean_rt")
n = len(agg)
agg["rt_rank"] = [min(math.floor(i / n * N_BINS) + 1, N_BINS) for i in range(n)]

# ── sort by position and write ────────────────────────────────────────────────
agg["chrom_order"] = agg["chr"].astype(int)
agg = agg.sort_values(["chrom_order", "win_start"])
agg["win_end"] = agg["win_start"] + WINDOW_SIZE

agg[["chr", "win_start", "win_end", "mean_rt", "n_rt_windows", "rt_rank"]].to_csv(
    OUTFILE, sep="\t", index=False
)
print(f"Done! Written to {OUTFILE}")

In [ ]:
# calculate c>a opportunity per replication timing window

from collections import defaultdict
import bisect
import gzip
import pyfastx
import pandas as pd

BED       = "./rt_windows.bed"
FASTA     = "hg38.fa.gz"
GIAB_BED  = "./giab_difficult_merged_oct27.bed"
AUTOSOMES = set(["chr" + str(i) for i in range(1, 23)])

# ── load giab ─────────────────────────────────────────────────────────────────
giab = defaultdict(list)
with open(GIAB_BED) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("chr"):
            continue
        fields = line.split("\t")
        if not fields[1].lstrip("-").isdigit():
            continue
        chrom, start, end = "chr" + fields[0], int(fields[1]), int(fields[2])
        giab[chrom].append((start, end))
for chrom in giab:
    giab[chrom].sort()

# precompute starts list per chrom for fast bisect
giab_starts = {chrom: [s for s, e in ivs] for chrom, ivs in giab.items()}

def get_giab_overlaps(chrom, win_start, win_end):
    """return list of (start, end) giab intervals overlapping this window."""
    starts = giab_starts.get(chrom, [])
    ivs    = giab.get(chrom, [])
    if not starts:
        return []
    # first interval that could overlap: its end > win_start
    # find rightmost start <= win_end
    right = bisect.bisect_right(starts, win_end)
    overlaps = []
    for i in range(right):
        s, e = ivs[i]
        if e > win_start:   # overlaps window
            overlaps.append((max(s, win_start), min(e, win_end)))
    return overlaps

# ── load reference ────────────────────────────────────────────────────────────
fa = pyfastx.Fasta(FASTA)

results = []
with open(BED) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        fields = line.split("\t")
        if not fields[1].lstrip("-").isdigit():
            continue
        chrom     = "chr" + fields[0]
        win_start = int(fields[1])
        win_end   = int(fields[2])

        if chrom not in AUTOSOMES:
            continue

        # fetch whole window in one call
        seq = bytearray(fa[chrom][win_start:win_end].seq.upper().encode())

        # mask giab positions in place (set to n)
        for ms, me in get_giab_overlaps(chrom, win_start, win_end):
            for i in range(ms - win_start, me - win_start):
                seq[i] = ord('N')

        seq = seq.decode()
        c_count      = seq.count("C")
        g_count      = seq.count("G")
        unmasked_bp  = sum(1 for b in seq if b != 'N')

        results.append({
            "chr":            chrom,
            "start":          win_start,
            "end":            win_end,
            "unmasked_bp":    unmasked_bp,
            "n_C":            c_count,
            "n_G":            g_count,
            "ca_opportunity": c_count + g_count
        })

df = pd.DataFrame(results)
df.to_csv("./rt_windows_ca_opportunity_giab_masked.tsv", sep="\t", index=False)
print(df.head())
print(f"\nTotal windows: {len(df)}")
print(f"Mean unmasked bp per window: {df['unmasked_bp'].mean():.0f}")
print(f"Mean C>A opportunity per window: {df['ca_opportunity'].mean():.1f}")

In [ ]:
# calculate c>g opportunity per replication timing window 


import pandas as pd
import pyfastx
from collections import defaultdict
import bisect

FASTA    = "hg38.fa.gz"
TSV      = "./chr8_cg.tsv"
GIAB_BED = "./giab_difficult_merged_oct27.bed"

# ── load giab ─────────────────────────────────────────────────────────────────
giab = defaultdict(list)
with open(GIAB_BED) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("chr"):
            continue
        fields = line.split("\t")
        if not fields[1].lstrip("-").isdigit():
            continue
        chrom, start, end = "chr" + fields[0], int(fields[1]), int(fields[2])
        giab[chrom].append((start, end))
for chrom in giab:
    giab[chrom].sort()
giab_starts = {chrom: [s for s, e in ivs] for chrom, ivs in giab.items()}

def get_giab_overlaps(chrom, win_start, win_end):
    starts = giab_starts.get(chrom, [])
    ivs    = giab.get(chrom, [])
    if not starts:
        return []
    right = bisect.bisect_right(starts, win_end)
    overlaps = []
    for i in range(right):
        s, e = ivs[i]
        if e > win_start:
            overlaps.append((max(s, win_start), min(e, win_end)))
    return overlaps

# ── load data and reference ───────────────────────────────────────────────────
df = pd.read_csv(TSV, sep="\t")
fa = pyfastx.Fasta(FASTA)

# ── count c>g opportunity with giab masked ────────────────────────────────────
cg_opps = []
for _, row in df.iterrows():
    chrom     = row["chrom"]
    win_start = int(row["start"])
    win_end   = int(row["end"])

    seq = bytearray(fa[chrom][win_start:win_end].seq.upper().encode())

    for ms, me in get_giab_overlaps(chrom, win_start, win_end):
        for i in range(ms - win_start, me - win_start):
            seq[i] = ord('N')

    seq_str = seq.decode()
    cg_opps.append(seq_str.count("C") + seq_str.count("G"))

df["cg_opportunity"] = cg_opps

print(df[["chrom", "start", "end", "cg_opportunity"]].to_string())
df.to_csv("./chr8_cg_opportunity.tsv", sep="\t", index=False)